# Sentiment Analysis Using Bag of Words and Random Forest

## Project Overview

Natural Language Processing (NLP) enables machines to understand, analyze, and extract meaningful information from human language. One of the most fundamental NLP tasks is **Sentiment Analysis**, which aims to determine whether a piece of text expresses a positive or negative opinion.

In this project, I build a complete sentiment analysis pipeline using classical NLP techniques and Machine Learning algorithms. The workflow covers the entire process from raw text preprocessing to feature extraction, model training, evaluation, and experiment documentation.

The primary objective of this project is not only to achieve high predictive performance but also to systematically investigate how different preprocessing techniques, feature engineering strategies, and machine learning models affect sentiment classification performance.

---

## Dataset

Dataset Link: **[IMBD Movie Reviews](https://www.kaggle.com/datasets/mwallerphunware/imbd-movie-reviews-for-binary-sentiment-analysis)**

The dataset consists of movie reviews labeled with their corresponding sentiment:

* Positive
* Negative

The reviews contain real-world natural language, making them suitable for evaluating text preprocessing techniques and machine learning models.

---

## Project Pipeline

The following stages are implemented throughout this project:

### 1. Text Preprocessing

* Contraction expansion
* Lowercasing
* Text cleaning using regular expressions
* Stopword removal with preserved negations
* Lemmatization
* Corpus construction

### 2. Feature Extraction

* Bag of Words (CountVectorizer)
* N-gram generation
* Vocabulary analysis
* Feature selection using `max_features`
* Rare-word filtering using `min_df`

### 3. Model Training

Different machine learning algorithms are evaluated and compared, including:

* Gaussian Naive Bayes
* Multinomial Naive Bayes
* Bernoulli Naive Bayes
* Logistic Regression
* Support Vector Machines (SVM)
* Decision Trees
* Random Forests

### 4. Model Evaluation

Performance is assessed using multiple metrics:

* Accuracy
* Precision
* Recall
* F1-Score
* ROC-AUC Score
* Confusion Matrix
* Cross-Validation Mean Accuracy
* Cross-Validation Standard Deviation

---

## Experimental Approach

Rather than training a single model, this notebook follows an experimentation-driven methodology.

For each model, multiple configurations are tested, including different values for:

* `max_features`
* `min_df`
* N-gram ranges
* Preprocessing strategies

The goal is to identify the most effective configuration while understanding the trade-offs between model complexity, computational cost, and predictive performance.

---

## Key Learning Objectives

Through this project, I aim to:

* Develop a deeper understanding of NLP preprocessing techniques.
* Compare the behavior of different machine learning algorithms on text data.
* Analyze how feature engineering impacts classification performance.
* Build reproducible NLP pipelines suitable for real-world applications.
* Establish strong baselines before moving toward advanced embedding and transformer-based approaches.

---

**Author:** Hazem Mohamed

**Role:** AI Engineer | Machine Learning Engineer | NLP Engineer

**Repository:** [NLP Experimentation Lab](https://github.com/Hazem1695/NLP-Experimentation-Lab)


# **Importing the Libraries**

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# **Data preprocessing**

## Data Cleaning Check Template
This template is designed to quickly assess the quality of any dataset before building machine learning models or performing analysis.

It provides a structured overview of the dataset by checking for common data issues such as:

- Missing values

- Duplicate rows

- Incorrect data types

- Outliers

- Distribution of numerical features

- Categorical feature consistency

**What This Template Does**

- Displays basic dataset information (shape, data types)

- Identifies missing values and duplicates

- Summarizes numerical and categorical features

- Detects potential outliers using the IQR method

- Highlights columns with low unique values for quick inspection

How to Use

1. Load your dataset using Pandas  

2. Call the function:

In [2]:
def data_quality_report(df):

    print("DATA QUALITY REPORT")
    
    # Print a separator line for better readability
    
    print("=" * 50)
    print("BASIC INFO")
    print("=" * 50)
    
    # Show general information about the dataset (columns, data types, non-null values)
    print(df.info())
    
    # Show number of rows and columns
    print("\n" + "=" * 50)
    print("SHAPE OF DATA")
    print("=" * 50)
    print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
    
    # Check for missing (null) values in each column
    print("\n" + "=" * 50)
    print("MISSING VALUES")
    print("=" * 50)
    missing = df.isnull().sum()
    
    # Display only columns that have missing values
    print(missing[missing > 0])
    
    # Check for duplicate rows
    print("\n" + "=" * 50)
    print("DUPLICATES")
    print("=" * 50)
    print(f"Duplicate rows: {df.duplicated().sum()}")
    
    # Display data types of each column
    print("\n" + "=" * 50)
    print("DATA TYPES")
    print("=" * 50)
    print(df.dtypes)
    
    # Summary statistics for numerical columns (mean, std, min, max, etc.)
    print("\n" + "=" * 50)
    print("NUMERICAL SUMMARY")
    print("=" * 50)
    print(df.describe())
    
    # Summary for categorical (object) columns
    print("\n" + "=" * 50)
    print("CATEGORICAL SUMMARY")
    print("=" * 50)
    print(df.describe(include=['object']))
    
    # Show unique values for columns with low number of distinct values
    # Useful for detecting categories, errors, or inconsistencies
    print("\n" + "=" * 50)
    print("UNIQUE VALUES (LOW CARDINALITY)")
    print("=" * 50)
    for col in df.columns:
        if df[col].nunique() < 10:  # Only show columns with few unique values
            print(f"{col}: {df[col].unique()}")
            
    # correlation
    print("\n" + "=" * 50)
    print("CORRELATION MATRIX")
    print("=" * 50)
    print(df.corr(numeric_only=True))
    
    # Detect outliers using the IQR (Interquartile Range) method
    print("\n" + "=" * 50)
    print("OUTLIERS CHECK (IQR METHOD)")
    print("=" * 50)
    
    # Loop through only numerical columns
    for col in df.select_dtypes(include=np.number).columns:
        Q1 = df[col].quantile(0.25)  # 25th percentile
        Q3 = df[col].quantile(0.75)  # 75th percentile
        IQR = Q3 - Q1  # Interquartile range
        
        # Count rows that fall outside the normal range
        outliers = df[(df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)]
        print(f"{col}: {len(outliers)} outliers")

## **Load dataset**
Apply Data Cleaning Check Template

In [ ]:
dataset = pd.read_csv('MovieReviewTrainingDatabase.csv')
data_quality_report(dataset)

DATA QUALITY REPORT
BASIC INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   sentiment  25000 non-null  object
 1   review     25000 non-null  object
dtypes: object(2)
memory usage: 390.8+ KB
None

SHAPE OF DATA
Rows: 25000, Columns: 2

MISSING VALUES
Series([], dtype: int64)

DUPLICATES
Duplicate rows: 96

DATA TYPES
sentiment    object
review       object
dtype: object

NUMERICAL SUMMARY
       sentiment                                             review
count      25000                                              25000
unique         2                                              24904
top     Positive  You do realize that you've been watching the E...
freq       12500                                                  3

CATEGORICAL SUMMARY
       sentiment                                             review
count      25000                 

## Duplicate Data Detection

In [4]:
duplicates = dataset[dataset.duplicated(subset=['review'], keep=False)]
duplicates.sort_values('review')

,sentiment,review
21186,Negative,"Back in his youth, the old man had wanted to..."
21877,Negative,"Back in his youth, the old man had wanted to..."
14734,Negative,'Dead Letter Office' is a low-budget film abou...
5519,Negative,'Dead Letter Office' is a low-budget film abou...
7011,Positive,".......Playing Kaddiddlehopper, Col San Fernan..."
...,...,...
2685,Negative,"in this movie, joe pesci slams dunks a basketb..."
22244,Positive,it's amazing that so many people that i know h...
14767,Positive,it's amazing that so many people that i know h...
12462,Negative,this movie begins with an ordinary funeral... ...


## Quantifying Duplicate Review Frequencies

In [5]:
review_counts = dataset['review'].value_counts()
print("Reviews appearing more than once:")
print((review_counts > 1).sum())
print("\nMaximum repetitions:")
print(review_counts.max())

Reviews appearing more than once:
92

Maximum repetitions:
3


## Removing Duplicate Reviews & Resetting Index
> **Note:** This cell drops the repeated rows we identified in the previous steps and cleanly resets the row indices for model training

In [6]:
print("Before:", len(dataset))
dataset = dataset.drop_duplicates()
print("After:", len(dataset))
dataset = dataset.reset_index(drop=True)

Before: 25000
After: 24904


## Library Installation
> **Note:** The `contractions` library is required to automatically expand shortcuts like *don't* to *do not* and *I'm* to *I am* during preprocessing.

In [7]:
!pip install contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.9 MB/s eta 0:00:00


## **Cleaning the texts**

In [8]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
import re
import contractions
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

base_stopwords = set(stopwords.words('english'))
negation_words = {'not', 'no', 'never'}
all_stopwords = base_stopwords - negation_words

wnl = WordNetLemmatizer()

corpus = []

for i in range(0, len(dataset)):
    review = dataset['review'][i]
    # Fix contractions
    review = contractions.fix(review)
    # Lowercase
    review = review.lower()
    
    review = re.sub(r'[^a-zA-Z\s]', ' ', review)
    # Split
    words = review.split()
    # Chained Lemmatization (Handles both Verbs 'v' and Nouns 'n')
    review = [wnl.lemmatize(wnl.lemmatize(word, pos='v'), pos='n') for word in words if word not in all_stopwords]
    review = ' '.join(review) 
    corpus.append(review)

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


## Preprocessing Verification
> **Note:** Pulling the first two rows directly as a memory array to confirm that our lowercasing, stopword stripping, and lemmatization pipeline worked correctly before feeding it into the vectorizer.

In [9]:
# Pull the data directly as a fast memory array
raw_samples = dataset['review'].head(2).values

for i in range(2):
    print(f"=== REVIEW #{i+1} ===")
    print(f"RAW:     {raw_samples[i]}\n") 
    print(f"CLEANED: {corpus[i]}")
    print("-" * 50)

=== REVIEW #1 ===
RAW:     With all this stuff going down at the moment with MJ i've started listening to his music, watching the odd documentary here and there, watched The Wiz and watched Moonwalker again. Maybe i just want to get a certain insight into this guy who i thought was really cool in the eighties just to maybe make up my mind whether he is guilty or innocent. Moonwalker is part biography, part feature film which i remember going to see at the cinema when it was originally released. Some of it has subtle messages about MJ's feeling towards the press and also the obvious message of drugs are bad m'kay.  Visually impressive but of course this is all about Michael Jackson so unless you remotely like MJ in anyway then you are going to hate this and find it boring. Some may call MJ an egotist for consenting to the making of this movie BUT MJ and most of his fans would say that he made it for the fans which if true is really nice of him.  The actual feature film bit when it final

# **Creating the Bag of Word model**

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
# Add ngram_range=(1, 2) so it automatically catches phrases like "not good" or "no clue"
cv = CountVectorizer(max_features=15000, ngram_range=(1, 2))
X = cv.fit_transform(corpus).toarray()
y = dataset.iloc[:, 0].values

In [11]:
len(X[0])

15000

# **Encoding Categorical data Using Label Encoding**

In [12]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y)

In [13]:
print(y)

[1 1 0 ... 0 0 1]


# Class Balance Check
> **Note:** Using NumPy to verify if our dataset is perfectly balanced between positive and negative reviews before splitting it into training and testing sets.

In [14]:
# This returns the unique classes and how many times they appear
classes, counts = np.unique(y, return_counts=True)
for c, count in zip(classes, counts):
    print(f"Class {c} contains {count}")

Class 0 contains 12432
Class 1 contains 12472


# **Splitting the dataset into the Training set and Test set**

In [15]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state = 0)

# Tuning Hyperparameters with HalvingRandomSearchCV

In [19]:
from sklearn.experimental import enable_halving_search_cv  # noqa
from sklearn.model_selection import HalvingRandomSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import time

# Initialize Random Forest
rf = RandomForestClassifier(random_state=0)

# Hyperparameter search space
param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'criterion': ['gini', 'entropy', 'log_loss'],
    'max_depth': [None, 10, 20, 30, 50],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True, False]
}

In [20]:
# Halving Random Search
halving_search = HalvingRandomSearchCV(
    estimator=rf,
    param_distributions=param_grid,
    factor=3,                # Reduce candidates by 3x each round
    resource='n_samples',    # Increase number of training samples each round
    max_resources='auto',
    cv=5,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    verbose=2
)


In [ ]:
# Train
halving_search.fit(X_train, y_train)

# Best model
print(halving_search.best_estimator_)
print(halving_search.best_params_)
print(halving_search.best_score_)

n_iterations: 7
n_required_iterations: 7
n_possible_iterations: 7
min_resources_: 20
max_resources_: 19923
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 996
n_resources: 20
Fitting 5 folds for each of 996 candidates, totalling 4980 fits
[CV] END bootstrap=False, criterion=entropy, max_depth=20, max_features=None, min_samples_leaf=2, min_samples_split=20, n_estimators=300; total time=   0.7s
[CV] END bootstrap=False, criterion=entropy, max_depth=20, max_features=None, min_samples_leaf=2, min_samples_split=20, n_estimators=300; total time=   0.6s
[CV] END bootstrap=True, criterion=log_loss, max_depth=10, max_features=log2, min_samples_leaf=1, min_samples_split=10, n_estimators=500; total time=   1.3s
[CV] END bootstrap=False, criterion=log_loss, max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=20, n_estimators=200; total time=   0.4s
[CV] END bootstrap=False, criterion=log_loss, max_depth=20, max_features=log2, min_samples_leaf=8, min_sa

# **Training the Random Forest model on the Training set**

In [16]:
from sklearn.ensemble import RandomForestClassifier
classifier = RandomForestClassifier(n_estimators=500, max_depth = 50, min_samples_leaf = 1, min_samples_split = 10, max_features = 'sqrt', criterion='entropy', bootstrap = False, random_state=0)
classifier.fit(X_train, y_train)

RandomForestClassifier(bootstrap=False, criterion='entropy', max_depth=50,
                       min_samples_split=10, n_estimators=500, random_state=0)

# **Predicting the Test set results**

In [17]:
y_pred = classifier.predict(X_test)
print(np.concatenate((y_pred.reshape(len(y_pred),1), y_test.reshape(len(y_test),1)),1))

[[1 1]
 [1 1]
 [1 0]
 ...
 [0 0]
 [0 0]
 [1 0]]


# **Evaluating the Model Performance**

In [18]:
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import cross_val_score

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nAccuracy Score:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nROC-AUC Score:")
print(roc_auc_score(y_test, y_pred))


accuracies = cross_val_score(estimator=classifier, X=X_train, y=y_train, cv=3)

print("\nMean Accuracy:")
print(accuracies.mean())

print("\nStandard Deviation:")
print(accuracies.std())

Confusion Matrix:
[[2090  430]
 [ 276 2185]]

Accuracy Score:
0.8582613932945192

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.83      0.86      2520
           1       0.84      0.89      0.86      2461

    accuracy                           0.86      4981
   macro avg       0.86      0.86      0.86      4981
weighted avg       0.86      0.86      0.86      4981


ROC-AUC Score:
0.8586077733273995

Mean Accuracy:
0.8583546654620289

Standard Deviation:
0.001998926642748576


# Random Forest Performance Analysis for Text Classification

# 1. Objective

The objective of this experiment was to investigate the effectiveness of the **Random Forest Classifier** for binary text classification using **Bag-of-Words (BoW)** features generated by **CountVectorizer**.

Random Forest is an ensemble learning algorithm that combines the predictions of multiple Decision Trees to improve generalization, reduce overfitting, and produce more stable predictions. Since single Decision Trees often struggle with sparse and high-dimensional text representations, this study evaluates whether ensemble learning can significantly improve classification performance.

The experiment focuses on answering the following questions:

* Can Random Forest outperform a single Decision Tree on sparse NLP data?
* How does the vocabulary size affect model performance?
* Does class balancing improve prediction quality?
* Is hyperparameter optimization beneficial?
* Why was **HalvingRandomSearchCV** selected instead of GridSearchCV or RandomizedSearchCV?

---

# 2. Experimental Setup

## Dataset

The dataset was preprocessed before training.

The preprocessing pipeline included:

* Text cleaning
* Duplicate removal
* Tokenization
* Bag-of-Words representation

Final evaluation was performed on **4,981 testing samples**.

---

## Feature Extraction

Documents were converted into numerical vectors using **CountVectorizer**.

Several vocabulary sizes were evaluated:

| Configuration | Parameters                        |
| ------------- | --------------------------------- |
| C1            | max_features = 5,000              |
| C2            | max_features = 10,000             |
| C3            | max_features = 15,000             |
| C4            | max_features = 20,000, min_df = 2 |

All experiments used:

* Unigrams + Bigrams (`ngram_range=(1,2)`)

---

# 3. Hyperparameter Optimization Strategy

Initially, manually selected hyperparameters were evaluated to establish a performance baseline.

After obtaining the baseline, hyperparameter optimization was performed.

Instead of using **GridSearchCV**, which becomes computationally expensive for Random Forests due to the large hyperparameter space, **HalvingRandomSearchCV** was selected.

HalvingRandomSearchCV progressively eliminates weak parameter configurations while allocating more computational resources to the most promising candidates. This strategy significantly reduces training time while still discovering high-quality hyperparameters.

Compared with exhaustive Grid Search, Halving Random Search provides a much better trade-off between computational efficiency and model performance, especially for ensemble methods such as Random Forest.

---

# 4. Hyperparameter Optimization

The optimization process explored combinations of:

* Number of trees (`n_estimators`)
* Maximum tree depth
* Minimum samples per split
* Minimum samples per leaf
* Feature selection strategy
* Bootstrap sampling
* Splitting criterion

The best-performing configuration was:

```python
RandomForestClassifier(
    n_estimators=500,
    max_depth=50,
    min_samples_leaf=1,
    min_samples_split=10,
    max_features="sqrt",
    criterion="entropy",
    bootstrap=False,
    random_state=0
)
```

---

# 5. Experimental Results

| Configuration                   |   Accuracy |    ROC-AUC |    CV Mean |     CV Std |
| ------------------------------- | ---------: | ---------: | ---------: | ---------: |
| Baseline + Class Weight         |     82.33% |     0.8241 |     82.13% |     0.0039 |
| Baseline (No Class Weight)      |     81.73% |     0.8182 |     82.06% |     0.0038 |
| Optimized RF (5K Features)      |     85.22% |     0.8525 |     84.93% |     0.0034 |
| Optimized RF (10K Features)     |     85.44% |     0.8548 |     85.13% |     0.0022 |
| **Optimized RF (15K Features)** | **85.83%** | **0.8586** | **85.84%** | **0.0020** |
| Optimized RF (20K Features)     |     85.79% |     0.8582 |     85.81% |     0.0043 |

---

# 6. Performance Analysis

## Effect of Ensemble Learning

Compared with the previously evaluated Decision Tree model, Random Forest produced a substantial improvement.

| Model         | Best Accuracy |
| ------------- | ------------: |
| Decision Tree |        74.44% |
| Random Forest |    **85.83%** |

The improvement exceeded **11 percentage points**, demonstrating the effectiveness of ensemble learning for high-dimensional NLP datasets.

Unlike a single Decision Tree, Random Forest combines hundreds of independent trees, reducing variance and producing significantly more stable decision boundaries.

---

## Effect of Hyperparameter Optimization

Hyperparameter optimization produced a clear performance improvement.

Baseline accuracy:

**82.33%**

↓

Optimized accuracy:

**85.83%**

This improvement confirms that carefully selecting tree depth, number of estimators, splitting constraints, and bootstrap strategy substantially enhances model generalization.

---

## Effect of Vocabulary Size

Increasing the vocabulary size consistently improved model performance.

| Vocabulary Size |   Accuracy |
| --------------: | ---------: |
|           5,000 |     85.22% |
|          10,000 |     85.44% |
|      **15,000** | **85.83%** |
|          20,000 |     85.79% |

The largest improvement occurred when moving from the baseline to richer vocabularies.

Beyond **15,000 features**, additional vocabulary contributed only marginal gains, indicating diminishing returns.

---

## Effect of Class Weight

The first experiment evaluated:

```python
class_weight="balanced"
```

Removing class balancing slightly reduced performance.

| Setting      |   Accuracy |
| ------------ | ---------: |
| Balanced     | **82.33%** |
| No balancing |     81.73% |

Although the improvement was modest, balanced class weighting produced better overall recall and class separation.

---

## Effect of Bootstrap Sampling

The optimized model selected:

```python
bootstrap=False
```

instead of the traditional bootstrap sampling.

For this dataset, training every tree using the complete training set produced slightly stronger decision boundaries and higher classification accuracy.

---

# 7. Precision and Recall Analysis

The best-performing Random Forest achieved:

### Class 0

* Precision = **0.88**
* Recall = **0.83**
* F1-score = **0.86**

### Class 1

* Precision = **0.84**
* Recall = **0.89**
* F1-score = **0.86**

The model demonstrated balanced performance across both classes.

Unlike Decision Trees, Random Forest maintained high precision while simultaneously achieving strong recall, indicating much better generalization.

---

# 8. ROC-AUC Analysis

ROC-AUC increased steadily throughout the experiments.

| Configuration       |    ROC-AUC |
| ------------------- | ---------: |
| Baseline            |     0.8241 |
| Optimized (5K)      |     0.8525 |
| Optimized (10K)     |     0.8548 |
| **Optimized (15K)** | **0.8586** |
| Optimized (20K)     |     0.8582 |

The improvement demonstrates that the optimized Random Forest became significantly better at distinguishing between the two target classes.

---

# 9. Cross-Validation Analysis

The best-performing model achieved:

* Mean Accuracy = **85.84%**
* Standard Deviation = **0.0020**

The extremely small standard deviation indicates highly consistent performance across different validation folds.

This suggests that the optimized Random Forest generalizes well and is not highly sensitive to the particular train-test split.

---

# 10. Best Model Configuration

The highest-performing configuration was:

```python
CountVectorizer(
    max_features=15000,
    ngram_range=(1,2)
)

RandomForestClassifier(
    n_estimators=500,
    max_depth=50,
    min_samples_leaf=1,
    min_samples_split=10,
    max_features="sqrt",
    criterion="entropy",
    bootstrap=False,
    random_state=0
)
```

Performance:

* Accuracy = **85.83%**
* Precision = **0.86**
* Recall = **0.86**
* F1-score = **0.86**
* ROC-AUC = **0.8586**
* Cross-Validation Accuracy = **85.84%**
* Cross-Validation Standard Deviation = **0.0020**

---

# 11. Comparison with Previously Evaluated Models

| Model                   | Best Accuracy |    ROC-AUC |
| ----------------------- | ------------: | ---------: |
| Logistic Regression     |    **88.92%** | **0.8894** |
| LinearSVC               |        87.71% |     0.8773 |
| Bernoulli Naive Bayes   |        87.66% |     0.8767 |
| Multinomial Naive Bayes |        87.56% |     0.8755 |
| **Random Forest**       |    **85.83%** | **0.8586** |
| Gaussian Naive Bayes    |        81.70% |     0.8156 |
| K-Nearest Neighbors     |        75.31% |     0.7523 |
| Decision Tree           |        74.44% |     0.7453 |

Random Forest substantially outperformed Decision Tree by leveraging ensemble learning, reducing overfitting, and improving predictive stability. However, it still lagged behind the best-performing linear models and Naive Bayes classifiers, which are particularly well suited to sparse Bag-of-Words representations.

---

# 12. Discussion

The experiments demonstrate that Random Forest effectively overcomes many of the weaknesses of a single Decision Tree by aggregating the predictions of hundreds of independent trees. This ensemble approach significantly improves robustness and generalization.

Hyperparameter optimization played a major role in achieving competitive performance. Increasing the number of trees, allowing deeper tree structures, and disabling bootstrap sampling all contributed to improved classification accuracy.

Nevertheless, despite these improvements, Random Forest did not surpass Logistic Regression or LinearSVC. This outcome is expected because Bag-of-Words features create extremely high-dimensional sparse vectors, where linear classifiers are particularly effective. Random Forests, while powerful, are generally less efficient at modeling sparse text features than linear decision boundaries.

---

# 13. Final Conclusion

This study evaluated the performance of the Random Forest Classifier for binary text classification using multiple Bag-of-Words feature configurations and extensive hyperparameter optimization.

The experiments showed that ensemble learning dramatically improved performance compared with a single Decision Tree, increasing accuracy from **74.44%** to **85.83%**. Hyperparameter optimization using **HalvingRandomSearchCV** enabled efficient exploration of the search space while significantly reducing computational cost compared with exhaustive search methods.

The best-performing model combined **CountVectorizer with 15,000 features** and an optimized **RandomForestClassifier** containing **500 decision trees** with entropy splitting, a maximum depth of 50, and no bootstrap sampling. This configuration achieved an **Accuracy of 85.83%**, an **ROC-AUC of 0.8586**, and a highly stable **Cross-Validation Accuracy of 85.84%**.

Although Random Forest proved to be a strong and reliable ensemble method, the experiments also confirmed that **Logistic Regression** and **LinearSVC** remain better suited for high-dimensional sparse text classification. Their simpler linear decision boundaries provide higher predictive performance, faster training, and better scalability, making them the preferred models for Bag-of-Words-based NLP tasks.
